In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb;

In [0]:
%sql
create table retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
dbutils.fs.ls('gs://databricksfolder/gold/')
# [FileInfo(path='gs://databricksfolder/gold/_delta_log/', name='_delta_log/', size=0, modificationTime=0)]
dbutils.fs.ls('gs://databricksfolder/silver/')
# [FileInfo(path='gs://databricksfolder/silver/_delta_log/', name='_delta_log/', size=0, modificationTime=0)]
dbutils.fs.ls('gs://databricksfolder/bronze/')
# [FileInfo(path='gs://databricksfolder/bronze/_delta_log/', name='_delta_log/', size=0, modificationTime=0)]

In [0]:
%sql
describe history retaildb.orders_bronze;
-- version	0
-- timestamp	2026-06-06T14:07:59.337Z
-- userId	218129261881966
-- userName	gauravmishra010@gmail.com
-- operation	CREATE TABLE
-- operationParameters	{"partitionBy":"[\"order_status\"]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null
-- notebook	{"notebookId":"3322675868849600"}
-- queryHistoryStatementId	835d63a4-4c00-40a3-af2d-4c464371edf2
-- clusterId	0606-083455-9yag8bsl-v2n
-- readVersion	null
-- isolationLevel	WriteSerializable
-- isBlindAppend	TRUE
-- operationMetrics	{}
-- userMetadata	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%fs
head gs://databricksfolder/raw/orders1.csv
order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:00.0,12111,CLOSED
2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,12111,CLOSED

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');
-- num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
-- 4,4,0


In [0]:
dbutils.fs.ls('gs://databricksfolder/bronze/')
# [FileInfo(path='gs://databricksfolder/bronze/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/bronze/order_status=CLOSED/', name='order_status=CLOSED/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/bronze/order_status=COMPLETE/', name='order_status=COMPLETE/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/bronze/order_status=PENDING_PAYMENT/', name='order_status=PENDING_PAYMENT/', size=0, modificationTime=0)]

In [0]:
%sql
select * from retaildb.orders_bronze limit 5;
-- order_id,order_date,customer_id,order_status,filename,createdon
-- 1,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 4,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z


In [0]:
%sql
describe history retaildb.orders_bronze;
-- version	1	0
-- timestamp	2026-06-06T14:13:08.747Z	2026-06-06T14:07:59.337Z
-- userId	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	COPY INTO	CREATE TABLE
-- operationParameters	{"statsOnLoad":"false"}	{"partitionBy":"[\"order_status\"]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null
-- notebook	{"notebookId":"3322675868849600"}	{"notebookId":"3322675868849600"}
-- queryHistoryStatementId	be82fce0-f150-40f4-ab89-0fe52c3c7ec2	835d63a4-4c00-40a3-af2d-4c464371edf2
-- clusterId	0606-083455-9yag8bsl-v2n	0606-083455-9yag8bsl-v2n
-- readVersion	0	null
-- isolationLevel	WriteSerializable	WriteSerializable
-- isBlindAppend	TRUE	TRUE
-- operationMetrics	{"numFiles":"3","numOutputRows":"4","numOutputBytes":"6069","numSkippedCorruptFiles":"0"}	{}
-- userMetadata	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
select * from table_changes('retaildb.orders_bronze', 1) limit 5;
-- order_id,order_date,customer_id,order_status,filename,createdon,_change_type,_commit_version,_commit_timestamp
-- 1,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 4,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z



In [0]:
# changes made in the bronze table from version 1 onwards to be merged to the silver table.
# first creating a temp of bronze table, then merge this to silver table

In [0]:
%sql
create or replace temporary view orders_bronze_changes
AS 
select * from table_changes('retaildb.orders_bronze', 1) where order_id > 0
AND customer_id > 0 and order_status IN ("CLOSED", "PENDING_PAYMENT");

In [0]:
%sql
select * from orders_bronze_changes limit 5;
-- order_id,order_date,customer_id,order_status,filename,createdon,_change_type,_commit_version,_commit_timestamp
-- 1,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 4,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z


In [0]:
%sql
select count(*) from orders_bronze_changes; -- 3
select count(*) from retaildb.orders_bronze; -- 4

In [0]:
%sql
merge into retaildb.orders_silver tgt
using orders_bronze_changes src on tgt.order_id = src.order_id
when matched
then
update set tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
WHEN not matched
then
insert(order_id, order_date, customer_id, order_status, createdon, modifiedon) values(order_id, order_date, customer_id, order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
-- num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
-- 3,0,0,3



In [0]:
dbutils.fs.ls('gs://databricksfolder/silver/')
# [FileInfo(path='gs://databricksfolder/silver/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/', name='order_status=CLOSED/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/', name='order_status=PENDING_PAYMENT/', size=0, modificationTime=0)]

In [0]:
dbutils.fs.ls('gs://databricksfolder/silver/order_status=CLOSED/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', name='part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', size=2234, modificationTime=1780755427427)]
dbutils.fs.ls('gs://databricksfolder/silver/order_status=PENDING_PAYMENT/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', name='part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', size=2194, modificationTime=1780755427437)]

In [0]:
%sql
select count(*) from retaildb.orders_bronze; -- 4
select count(*) from orders_bronze_changes; -- 3
select count(*) from retaildb.orders_silver; -- 3

In [0]:
dbutils.fs.ls('gs://databricksfolder/gold/')
# [FileInfo(path='gs://databricksfolder/gold/_delta_log/', name='_delta_log/', size=0, modificationTime=0)]

In [0]:
%sql
insert overwrite table retaildb.orders_gold
select customer_id, order_status, order_year, count(order_id) as num_orders
from retaildb.orders_silver group by 1, 2, 3;
-- num_affected_rows,num_inserted_rows
-- 2,2


In [0]:
%sql
select * from retaildb.orders_gold;
-- customer_id,order_status,order_year,num_orders
-- 12111,CLOSED,2013,2
-- 12111,PENDING_PAYMENT,2013,1

In [0]:
dbutils.fs.ls('gs://databricksfolder/gold/')
# [FileInfo(path='gs://databricksfolder/gold/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/gold/part-00000-a9a61855-fe51-43bc-b166-981734a2914b.c000.snappy.parquet', name='part-00000-a9a61855-fe51-43bc-b166-981734a2914b.c000.snappy.parquet', size=1352, modificationTime=1780755579714)]

In [0]:
%sql
select count(*) from retaildb.orders_bronze; -- 4
select count(*) from orders_bronze_changes; -- 3
select count(*) from retaildb.orders_silver; -- 3
select count(*) from retaildb.orders_gold; -- 2

In [0]:
# upload new file: orders_update
dbutils.fs.ls('gs://databricksfolder/raw/')
# [FileInfo(path='gs://databricksfolder/raw/orders1.csv', name='orders1.csv', size=203, modificationTime=1780755144744),
#  FileInfo(path='gs://databricksfolder/raw/orders_update.csv', name='orders_update.csv', size=212, modificationTime=1780755835864)]

In [0]:
%fs
head gs://databricksfolder/raw/orders_update.csv
order_id,order_date,customer_id,order_status
3,2013-07-25 00:00:00.0,12111,CLOSED
4,2013-07-25 00:00:00.0,8827,PENDING
11101,2013-07-25 00:00:00.0,12256,PROCESSING
11102,2013-07-25 00:00:00.0,7790,PENDING_PAYMENT

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');
-- num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
-- 4,4,0


In [0]:
# if we run the same copy into commands for the same exisitng file under raw folder, it will not copy the same existing records to our tables.

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');
-- num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
-- 0,0,0


In [0]:
%sql
select * from retaildb.orders_bronze;
-- order_id,order_date,customer_id,order_status,filename,createdon
-- 1,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 4,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 11102,2013-07-25 00:00:00.0,7790,PENDING_PAYMENT,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z
-- 11101,2013-07-25 00:00:00.0,12256,PROCESSING,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z
-- 2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z
-- 4,2013-07-25 00:00:00.0,8827,PENDING,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z
-- 3,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z

In [0]:
%sql
create or replace temporary view orders_bronze_changes
AS 
select * from table_changes('retaildb.orders_bronze', 1) where order_id > 0
AND customer_id > 0 and order_status IN ("CLOSED", "PENDING_PAYMENT");

In [0]:
%sql
select * from orders_bronze_changes where order_id in (3, 4, 11101, 11102) limit 5;
-- order_id,order_date,customer_id,order_status,filename,createdon,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders1.csv,2026-06-06T14:12:59.550Z,insert,1,2026-06-06T14:13:08.747Z
-- 11102,2013-07-25 00:00:00.0,7790,PENDING_PAYMENT,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z,insert,2,2026-06-06T14:26:42.479Z
-- 3,2013-07-25 00:00:00.0,12111,CLOSED,gs://databricksfolder/raw/orders_update.csv,2026-06-06T14:26:32.890Z,insert,2,2026-06-06T14:26:42.479Z


In [0]:
%sql
select count(*) from orders_bronze_changes limit 5; -- 5

In [0]:
%sql
select * from retaildb.orders_silver;
-- order_id,order_date,customer_id,order_status,order_year,order_month,createdon,modifiedon
-- 1,2013-07-25,12111,CLOSED,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:16:59.241Z
-- 4,2013-07-25,12111,CLOSED,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:16:59.241Z
-- 2,2013-07-25,12111,PENDING_PAYMENT,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:16:59.241Z


In [0]:
%fs
head gs://databricksfolder/raw/orders1.csv
order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:00.0,12111,CLOSED
2,2013-07-25 00:00:00.0,12111,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,12111,CLOSED

In [0]:
%fs
head gs://databricksfolder/raw/orders_update.csv
order_id,order_date,customer_id,order_status
3,2013-07-25 00:00:00.0,12111,CLOSED
4,2013-07-25 00:00:00.0,8827,PENDING
11101,2013-07-25 00:00:00.0,12256,PROCESSING
11102,2013-07-25 00:00:00.0,7790,PENDING_PAYMENT


In [0]:
dbutils.fs.ls('gs://databricksfolder/silver')
# [FileInfo(path='gs://databricksfolder/silver/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/', name='order_status=CLOSED/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/', name='order_status=PENDING_PAYMENT/', size=0, modificationTime=0)]
dbutils.fs.ls('gs://databricksfolder/silver/order_status=CLOSED/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', name='part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', size=2234, modificationTime=1780755427427)]
dbutils.fs.ls('gs://databricksfolder/silver/order_status=PENDING_PAYMENT/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', name='part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', size=2194, modificationTime=1780755427437)]

In [0]:
%sql
merge into retaildb.orders_silver tgt
using orders_bronze_changes src on tgt.order_id = src.order_id
when matched
then
update set tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
WHEN not matched
then
insert(order_id, order_date, customer_id, order_status, createdon, modifiedon) values(order_id, order_date, customer_id, order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
-- num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
-- 5,3,0,2



In [0]:
dbutils.fs.ls('gs://databricksfolder/silver')
# [FileInfo(path='gs://databricksfolder/silver/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/_change_data/', name='_change_data/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/', name='order_status=CLOSED/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/', name='order_status=PENDING_PAYMENT/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/silver/deletion_vector_bce44014-fc7f-4a1c-96a5-f096b7834756.bin', name='deletion_vector_bce44014-fc7f-4a1c-96a5-f096b7834756.bin', size=173, modificationTime=1780757633061)]

dbutils.fs.ls('gs://databricksfolder/silver/order_status=CLOSED/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/part-00001-0509ad61-b5cf-43cf-b885-57b40ac0419c.c000.snappy.parquet', name='part-00001-0509ad61-b5cf-43cf-b885-57b40ac0419c.c000.snappy.parquet', size=2632, modificationTime=1780757639396),
#  FileInfo(path='gs://databricksfolder/silver/order_status=CLOSED/part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', name='part-00001-409b6d1c-d830-429f-8670-3150e0f965cb.c000.snappy.parquet', size=2234, modificationTime=1780755427427)]

dbutils.fs.ls('gs://databricksfolder/silver/order_status=PENDING_PAYMENT/')
# [FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', name='part-00000-6b26b32a-4ba6-4753-9499-8f5657e52e31.c000.snappy.parquet', size=2194, modificationTime=1780755427437),
#  FileInfo(path='gs://databricksfolder/silver/order_status=PENDING_PAYMENT/part-00000-81fabf40-da9a-450f-8743-6043fc0440da.c000.snappy.parquet', name='part-00000-81fabf40-da9a-450f-8743-6043fc0440da.c000.snappy.parquet', size=2629, modificationTime=1780757638759)]


In [0]:
%sql
select * from retaildb.orders_silver
-- order_id,order_date,customer_id,order_status,order_year,order_month,createdon,modifiedon
-- 3,2013-07-25,12111,CLOSED,2013,7,2026-06-06T14:53:41.739Z,2026-06-06T14:53:41.739Z
-- 4,2013-07-25,12111,CLOSED,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:53:41.739Z
-- 1,2013-07-25,12111,CLOSED,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:53:41.739Z
-- 2,2013-07-25,12111,PENDING_PAYMENT,2013,7,2026-06-06T14:16:59.241Z,2026-06-06T14:53:41.739Z
-- 11102,2013-07-25,7790,PENDING_PAYMENT,2013,7,2026-06-06T14:53:41.739Z,2026-06-06T14:53:41.739Z


In [0]:
dbutils.fs.ls('gs://databricksfolder/gold/')
# [FileInfo(path='gs://databricksfolder/gold/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/gold/part-00000-a9a61855-fe51-43bc-b166-981734a2914b.c000.snappy.parquet', name='part-00000-a9a61855-fe51-43bc-b166-981734a2914b.c000.snappy.parquet', size=1352, modificationTime=1780755579714)]

In [0]:
%sql
select * from retaildb.orders_silver;

In [0]:
%sql
insert overwrite table retaildb.orders_gold
select customer_id, order_status, order_year, count(order_id) as num_orders
from retaildb.orders_silver group by 1, 2, 3;
-- num_affected_rows,num_inserted_rows
-- 2,2
